# 4. MODEL TRAINING
## Daily Customer Churn Predictor · VivaMarket Brasil

---

**INPUT:** `../data/processed/churn_features_YYYYMMDD.parquet`

*The customer-snapshot feature matrix created in NB03.*

**OUTPUT:** `../models/churn_model_YYYYMMDD.joblib` and `../data/processed/churn_predictions_YYYYMMDD.parquet`

*A trained churn model, comparative benchmark metrics and scored validation/test customer snapshots.*

---
## 4.1. STARTING SITUATION

NB03 transformed raw customer history into a modeling-ready snapshot matrix. The next task is to test whether those engineered features can rank churn risk in a way that is operationally useful for VivaMarket Brasil.

This notebook therefore focuses on **temporal model training**, not just on fitting algorithms. The project needs a comparison between simple baselines and a stronger gradient-boosting model, while respecting the monthly chronology of the snapshot data and the strong class imbalance observed in the churn label.

---
## 4.2. NOTEBOOK OBJECTIVE

- **Business objective:** identify the model that best prioritizes customers for retention campaigns under the project risk framework (High / Medium / Low).
- **Analytical objective:** compare temporal baselines, select the best candidate using imbalance-aware ranking metrics and persist scored outputs for downstream diagnostics and campaign design.

---
## 4.3. INITIAL SETUP

**What is done**

We load the required libraries for model training, scoring, artifact persistence and temporal evaluation.

**Why it is done**

NB04 must remain reproducible and explicit about the training flow, especially because the churn label is heavily imbalanced and model choice should be auditable.

**Expected result**

A stable environment with logging, resolved project paths and output folders ready for trained models and prediction files.

In [1]:
import logging
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    force=True,
)
logger = logging.getLogger('nb04_model_training')
logger.info('NB04 started: model training.')

2026-05-02 00:26:39,741 | INFO | NB04 started: model training.


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

feature_candidates = sorted(PROCESSED_DIR.glob('churn_features_*.parquet'))
if not feature_candidates:
    raise FileNotFoundError('No churn feature parquet found in data/processed.')

feature_path = feature_candidates[-1]
run_date_tag = datetime.now(ZoneInfo('Europe/Paris')).strftime('%Y%m%d')
model_output_path = MODELS_DIR / f'churn_model_{run_date_tag}.joblib'
prediction_output_path = PROCESSED_DIR / f'churn_predictions_{run_date_tag}.parquet'
metrics_output_path = PROCESSED_DIR / f'churn_model_metrics_{run_date_tag}.csv'

logger.info('Feature path resolved at %s', feature_path)
logger.info('Model output path resolved at %s', model_output_path)
logger.info('Prediction output path resolved at %s', prediction_output_path)

2026-05-02 00:26:39,750 | INFO | Feature path resolved at /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_features_20260502.parquet


2026-05-02 00:26:39,750 | INFO | Model output path resolved at /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/models/churn_model_20260502.joblib


2026-05-02 00:26:39,751 | INFO | Prediction output path resolved at /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_predictions_20260502.parquet


---
## 4.4. DATA LOADING AND MODELING SPLIT

**What is done**

We load the feature matrix, define a chronological train/validation/test split by snapshot date and prepare the list of modeling variables.

**Why it is done**

A churn model must be validated on future periods, not on shuffled rows, because the real production use case scores customers forward in time. The split therefore mirrors how the model would face unseen future snapshots.

**Expected result**

Three temporally ordered datasets with leak-free predictors, plus a documented view of class imbalance across the split.

In [3]:
feature_df = pd.read_parquet(feature_path)
feature_df['snapshot_date'] = pd.to_datetime(feature_df['snapshot_date'])
feature_df = feature_df.sort_values(['snapshot_date', 'customer_unique_id']).reset_index(drop=True)

snapshot_order = sorted(feature_df['snapshot_key'].unique())
train_snapshot_keys = snapshot_order[:10]
validation_snapshot_keys = snapshot_order[10:13]
test_snapshot_keys = snapshot_order[13:]

train_df = feature_df[feature_df['snapshot_key'].isin(train_snapshot_keys)].copy()
validation_df = feature_df[feature_df['snapshot_key'].isin(validation_snapshot_keys)].copy()
test_df = feature_df[feature_df['snapshot_key'].isin(test_snapshot_keys)].copy()

leakage_columns = [
    'customer_unique_id',
    'snapshot_key',
    'snapshot_date',
    'first_purchase_timestamp',
    'last_purchase_timestamp',
    'future_orders_90d',
    'future_revenue_90d',
    'churn_90d_label',
]
feature_columns = [column for column in feature_df.columns if column not in leakage_columns]

X_train = pd.get_dummies(train_df[feature_columns], columns=['customer_state'], dtype=float)
X_validation = pd.get_dummies(validation_df[feature_columns], columns=['customer_state'], dtype=float)
X_test = pd.get_dummies(test_df[feature_columns], columns=['customer_state'], dtype=float)

X_validation = X_validation.reindex(columns=X_train.columns, fill_value=0.0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0.0)

y_train = train_df['churn_90d_label'].astype(int)
y_validation = validation_df['churn_90d_label'].astype(int)
y_test = test_df['churn_90d_label'].astype(int)

split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'snapshots': [
        ', '.join(train_snapshot_keys),
        ', '.join(validation_snapshot_keys),
        ', '.join(test_snapshot_keys),
    ],
    'rows': [len(train_df), len(validation_df), len(test_df)],
    'churn_rate': [y_train.mean(), y_validation.mean(), y_test.mean()],
})
logger.info('Temporal split ready with %s train rows, %s validation rows and %s test rows.', len(train_df), len(validation_df), len(test_df))
split_summary

2026-05-02 00:26:42,620 | INFO | Temporal split ready with 108986 train rows, 59922 validation rows and 60421 test rows.


,split,snapshots,rows,churn_rate
0,train,"20170401, 20170501, 20170601, 20170701, 201708...",108986,0.990155
1,validation,"20180201, 20180301, 20180401",59922,0.990671
2,test,"20180501, 20180601, 20180701",60421,0.992370


---
## 4.5. BENCHMARK MODEL SET

**What is done**

We define three candidate models: logistic regression, random forest and XGBoost.

**Why it is done**

The project needs both interpretability-friendly baselines and a stronger nonlinear model. This prevents us from assuming that gradient boosting is best without comparison.

**Expected result**

A benchmark set ready for training under the same temporal split and comparable evaluation rules.

In [4]:
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / max(positive_count, 1)

logistic_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        max_iter=500,
        class_weight='balanced',
        solver='lbfgs',
        random_state=42,
    )),
])

random_forest_model = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    min_samples_leaf=20,
    class_weight='balanced_subsample',
    n_jobs=-1,
    random_state=42,
)

xgboost_model = XGBClassifier(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=4,
)

model_registry = {
    'logistic_regression': logistic_pipeline,
    'random_forest': random_forest_model,
    'xgboost': xgboost_model,
}

pd.DataFrame({
    'metric': ['train_positive_rows', 'train_negative_rows', 'scale_pos_weight'],
    'value': [positive_count, negative_count, scale_pos_weight],
})

,metric,value
0,train_positive_rows,107913.000000
1,train_negative_rows,1073.000000
2,scale_pos_weight,0.009943


---
## 4.6. TEMPORAL TRAINING AND VALIDATION

**What is done**

We fit each candidate model on the training snapshots and evaluate validation performance using ranking-oriented and threshold-aware metrics.

**Why it is done**

Because churn operations care most about whom to contact first, the evaluation must prioritize ranking quality, not just overall accuracy. Precision at the top of the score distribution is especially relevant for expensive retention actions.

**Expected result**

A model-comparison table that identifies the best candidate for downstream test scoring and campaign activation.

In [5]:
def precision_at_top_k(y_true: pd.Series, scores: np.ndarray, fraction: float = 0.10) -> float:
    evaluation_df = pd.DataFrame({'y_true': y_true.to_numpy(), 'score': scores})
    evaluation_df = evaluation_df.sort_values('score', ascending=False).reset_index(drop=True)
    cutoff = max(int(np.ceil(len(evaluation_df) * fraction)), 1)
    top_slice = evaluation_df.head(cutoff)
    return float(top_slice['y_true'].mean())


def evaluate_scores(y_true: pd.Series, scores: np.ndarray, split_name: str, model_name: str) -> dict:
    return {
        'model_name': model_name,
        'split': split_name,
        'roc_auc': roc_auc_score(y_true, scores),
        'average_precision': average_precision_score(y_true, scores),
        'precision_at_top_5pct': precision_at_top_k(y_true, scores, 0.05),
        'precision_at_top_10pct': precision_at_top_k(y_true, scores, 0.10),
        'mean_score': float(np.mean(scores)),
    }

benchmark_results = []
validation_scores = {}
trained_models = {}

for model_name, model in model_registry.items():
    logger.info('Training model: %s', model_name)
    model.fit(X_train, y_train)
    trained_models[model_name] = model

    validation_probability = model.predict_proba(X_validation)[:, 1]
    validation_scores[model_name] = validation_probability
    benchmark_results.append(evaluate_scores(y_validation, validation_probability, 'validation', model_name))

benchmark_df = pd.DataFrame(benchmark_results).sort_values(
    ['average_precision', 'precision_at_top_10pct', 'roc_auc'],
    ascending=False,
).reset_index(drop=True)
benchmark_df

2026-05-02 00:26:42,643 | INFO | Training model: logistic_regression


2026-05-02 00:26:59,292 | INFO | Training model: random_forest


2026-05-02 00:27:18,851 | INFO | Training model: xgboost


,model_name,split,roc_auc,average_precision,precision_at_top_5pct,precision_at_top_10pct,mean_score
0,xgboost,validation,0.617188,0.993500,0.995329,0.994660,0.628477
1,random_forest,validation,0.583369,0.992943,0.994995,0.994827,0.638371
2,logistic_regression,validation,0.597083,0.992421,0.992326,0.993326,0.542198


---
## 4.7. BEST MODEL SELECTION AND TEST SCORING

**What is done**

We select the strongest validation model, score the held-out test snapshots and create operational risk tiers using the retention thresholds already defined for the project.

**Why it is done**

The business does not need raw probabilities alone. It needs a ranked and tiered customer list that can flow into campaign logic such as High (>70%), Medium (40–70%) and Low (<40%) risk actions.

**Expected result**

A scored test set with model probabilities, risk tiers and a persisted best model artifact ready for downstream diagnostics and orchestration notebooks.

In [6]:
best_model_name = benchmark_df.iloc[0]['model_name']
best_model = trained_models[best_model_name]

test_probability = best_model.predict_proba(X_test)[:, 1]
benchmark_results.append(evaluate_scores(y_test, test_probability, 'test', best_model_name))
benchmark_df = pd.DataFrame(benchmark_results)

scored_test_df = test_df[[
    'customer_unique_id',
    'snapshot_key',
    'snapshot_date',
    'recency_days',
    'total_orders',
    'total_payment_value',
    'orders_30d',
    'orders_90d',
    'churn_90d_label',
]].copy()
scored_test_df['churn_probability'] = test_probability
scored_test_df['risk_tier'] = pd.cut(
    scored_test_df['churn_probability'],
    bins=[-np.inf, 0.40, 0.70, np.inf],
    labels=['LOW', 'MEDIUM', 'HIGH'],
)
scored_test_df['selected_model'] = best_model_name

joblib.dump({
    'model_name': best_model_name,
    'model': best_model,
    'feature_columns': X_train.columns.tolist(),
    'train_snapshot_keys': train_snapshot_keys,
    'validation_snapshot_keys': validation_snapshot_keys,
    'test_snapshot_keys': test_snapshot_keys,
}, model_output_path)

scored_test_df.to_parquet(prediction_output_path, index=False)
benchmark_df.to_csv(metrics_output_path, index=False)

logger.info('Best validation model: %s', best_model_name)
logger.info('Test predictions saved to %s', prediction_output_path)
logger.info('Model artifact saved to %s', model_output_path)
scored_test_df.head()

2026-05-02 00:27:22,279 | INFO | Best validation model: xgboost


2026-05-02 00:27:22,279 | INFO | Test predictions saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_predictions_20260502.parquet


2026-05-02 00:27:22,280 | INFO | Model artifact saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/models/churn_model_20260502.joblib


,customer_unique_id,snapshot_key,snapshot_date,recency_days,total_orders,total_payment_value,orders_30d,orders_90d,churn_90d_label,churn_probability,risk_tier,selected_model
168908,0004bd2a26a76fe21f786e4fbd80607f,20180501,2018-05-01,26,1,166.98,1.0,1,1,0.558454,MEDIUM,xgboost
168909,00050ab1314c0e55a6ca13cf7181fecf,20180501,2018-05-01,11,1,35.38,1.0,1,1,0.572675,MEDIUM,xgboost
168910,00053a61a98854899e70ed204dd4bafe,20180501,2018-05-01,62,1,419.18,0.0,1,1,0.896241,HIGH,xgboost
168911,0005ef4cd20d2893f0d9fbd94d3c0d97,20180501,2018-05-01,50,1,129.76,0.0,1,1,0.770620,HIGH,xgboost
168912,00090324bbad0e9342388303bb71ba0a,20180501,2018-05-01,38,1,63.66,0.0,1,1,0.708131,HIGH,xgboost


---
## 4.8. CAMPAIGN-READINESS SUMMARY

**What is done**

We summarize campaign volumes and priority tiers in the scored test set.

**Why it is done**

The retention strategy document already defines differentiated actions by risk band, so this summary translates modeling output into operational workload and opportunity size.

**Expected result**

A compact operational view of how many customer-snapshot cases would enter High, Medium and Low risk treatment under the selected model.

In [7]:
campaign_summary = (
    scored_test_df.groupby('risk_tier', observed=False)
    .agg(
        customers_n=('customer_unique_id', 'nunique'),
        rows_n=('customer_unique_id', 'size'),
        avg_churn_probability=('churn_probability', 'mean'),
        observed_churn_rate=('churn_90d_label', 'mean'),
        avg_total_payment_value=('total_payment_value', 'mean'),
    )
    .reset_index()
    .sort_values('risk_tier')
)
campaign_summary

,risk_tier,customers_n,rows_n,avg_churn_probability,observed_churn_rate,avg_total_payment_value
0,LOW,2613,3524,0.315564,0.978434,213.433984
1,MEDIUM,23845,38284,0.581426,0.992817,142.717869
2,HIGH,12484,18613,0.774315,0.994090,216.349209


In [8]:
top_feature_snapshot = pd.DataFrame({
    'feature': X_train.columns,
    'xgboost_importance': trained_models['xgboost'].feature_importances_ if 'xgboost' in trained_models else np.nan,
    'random_forest_importance': trained_models['random_forest'].feature_importances_ if 'random_forest' in trained_models else np.nan,
})

feature_importance_view = top_feature_snapshot.sort_values('xgboost_importance', ascending=False).head(20)
feature_importance_view

,feature,xgboost_importance,random_forest_importance
0,total_orders,0.105856,0.012739
11,distinct_categories_total,0.074619,0.013941
76,is_repeat_customer,0.050589,0.014580
62,distinct_products_180d,0.025090,0.007601
63,distinct_categories_180d,0.021350,0.012276
56,orders_180d,0.019756,0.009397
10,distinct_products_total,0.018901,0.007952
22,avg_review_score_30d,0.016597,0.010113
59,revenue_180d,0.011387,0.038401
88,customer_state_BA,0.011373,0.003066


---
## 4.9. NOTEBOOK CLOSURE

The modeling stage now has a trained benchmark and a selected best model under a temporal split. Three takeaways matter most for the next notebooks:

1. **ranking quality matters more than raw accuracy**, because retention budgets focus on the highest-risk part of the customer base;
2. **class imbalance remains a structural challenge**, so later evaluation should emphasize precision-recall, campaign coverage and business lift;
3. **risk tiers are now explicit**, which means NB05 and NB06 can evaluate diagnostics and explainability in terms that already connect to the retention-action strategy.

The next notebook should therefore test model stability in more depth, inspect threshold behavior and quantify how reliable the selected ranking is across future snapshots.